# Kernel Quality & Sample Diversity Evaluation

Analyzes the samples generated for each KernelBench problem in a single run folder.
Set `RUN_DIR` in the **Config** cell, then run all cells.

**Stages:**
1. Functional summary — compile/correct rates, runtime spread, error diversity
2. AST tree-edit distance — structural similarity between samples
3. Code-embedding cosine similarity — semantic similarity (jina-code-embeddings-1.5b)


## ⚙️ Config
*Only cell you need to edit per run.*

In [1]:
from pathlib import Path

# ─────────────────────────────────────────────────────────────────────────────
# ▶  CONFIGURE — change RUN_DIR to the run folder you want to analyze
# ─────────────────────────────────────────────────────────────────────────────
RUN_DIR = Path("/home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs"
               "/Qwen3-Coder-Next_level1_triton")

RUN_STAGE = {1: True, 2: True, 3: False}   # toggle individual stages

EMBED_MODEL = "jinaai/jina-code-embeddings-0.5b"

# Stage 3 device: auto-detect GPU, fall back to CPU
try:
    import torch as _torch
    DEVICE = "cuda" if _torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

OUT_DIR = RUN_DIR / "analysis"
OUT_DIR.mkdir(exist_ok=True)

print(f"RUN_DIR : {RUN_DIR}")
print(f"OUT_DIR : {OUT_DIR}")
print(f"Device  : {DEVICE}")
print(f"Stages  : {RUN_STAGE}")


RUN_DIR : /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton
OUT_DIR : /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis
Device  : cpu
Stages  : {1: True, 2: True, 3: False}


## Setup & Data Loader
Parses the run directory, attaches eval metadata where available.

In [2]:
import re, json, ast, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

try:
    import yaml
except ImportError:
    yaml = None

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 100
sys.setrecursionlimit(10_000)

# ── kernel filename pattern ───────────────────────────────────────────────────
_KERNEL_RE = re.compile(r"level_(\d+)_problem_(\d+)_sample_(\d+)_kernel\.py")


def load_run(run_dir: Path):
    """Load a KernelBench run directory.

    Returns
    -------
    problems : {problem_id: {sample_id: {code, eval, level}}}
    df       : flat pandas DataFrame (one row per sample)
    config   : generation_config.yaml as dict
    """
    run_dir = Path(run_dir)
    assert run_dir.exists(), f"Run dir not found: {run_dir}"

    # 1. generation config
    cfg_path = run_dir / "generation_config.yaml"
    config = {}
    if cfg_path.exists() and yaml is not None:
        with open(cfg_path) as f:
            config = yaml.safe_load(f) or {}

    # 2. eval results (may be absent in some run folders)
    eval_path = run_dir / "eval_results.json"
    eval_results: dict = {}
    if eval_path.exists():
        with open(eval_path) as f:
            eval_results = json.load(f)

    # 3. enumerate kernel files — sample ids can be non-contiguous
    problems: dict = {}
    for kp in sorted(run_dir.glob("*.py")):
        m = _KERNEL_RE.match(kp.name)
        if not m:
            continue
        level, prob_id, samp_id = int(m.group(1)), int(m.group(2)), int(m.group(3))
        code = kp.read_text()

        # attach eval entry when available
        ev = None
        for entry in eval_results.get(str(prob_id), []):
            if entry.get("sample_id") == samp_id:
                ev = entry
                break

        problems.setdefault(prob_id, {})[samp_id] = {
            "code": code, "eval": ev, "level": level
        }

    # 4. flat DataFrame
    rows = []
    for prob_id, samples in problems.items():
        for samp_id, data in samples.items():
            ev   = data["eval"] or {}
            rs   = ev.get("runtime_stats") or {}
            meta = ev.get("metadata") or {}
            rows.append({
                "problem_id":  prob_id,
                "sample_id":   samp_id,
                "level":       data["level"],
                "code":        data["code"],
                "code_len":    len(data["code"]),
                "has_eval":    data["eval"] is not None,
                "compiled":    ev.get("compiled"),
                "correctness": ev.get("correctness"),
                "runtime_ms":  ev.get("runtime"),
                "rt_mean":     rs.get("mean"),
                "rt_std":      rs.get("std"),
                "rt_min":      rs.get("min"),
                "rt_max":      rs.get("max"),
                "rt_trials":   rs.get("num_trials"),
                "error_name":  meta.get("runtime_error_name") or meta.get("error"),
                "error_msg":   meta.get("runtime_error"),
            })

    df = (pd.DataFrame(rows)
            .sort_values(["problem_id", "sample_id"])
            .reset_index(drop=True))
    return problems, df, config


# ── load ──────────────────────────────────────────────────────────────────────
problems, df, config = load_run(RUN_DIR)

has_eval = bool(df["has_eval"].any())
print(f"Loaded {len(df)} samples across {df['problem_id'].nunique()} problems")
print(f"  model   : {config.get('model_name', '?')}")
print(f"  level   : {config.get('level', '?')}")
print(f"  backend : {config.get('backend', '?')}")
print(f"  Eval results present: {has_eval}")
if has_eval:
    n = int(df["compiled"].notna().sum())
    print(f"  Compiled : {int(df['compiled'].sum())}/{n}")
    print(f"  Correct  : {int(df['correctness'].sum())}/{n}")

# flags for aggregate report
_s1_done = _s2_done = _s3_done = False
s1 = s2_df = s3_df = stage2_results = stage3_results = None


Loaded 1000 samples across 100 problems
  model   : Qwen/Qwen3-Coder-Next
  level   : 1
  backend : triton
  Eval results present: True
  Compiled : 884/1000
  Correct  : 161/1000


/home/jan-bosenius/PycharmProjects/GuidedResearch/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Stage 1 — Functional Summary
Per-problem: compile/correct counts, runtime stats (correct samples only), error diversity.

In [3]:
# ─── Stage 1: Functional Summary ─────────────────────────────────────────────
if not RUN_STAGE[1]:
    print("Stage 1 skipped (RUN_STAGE[1]=False)")
elif not has_eval:
    print("⚠️  No eval_results.json found — Stage 1 requires eval data. Skipping.")
else:
    rows1 = []
    for prob_id in tqdm(sorted(problems.keys()), desc="Stage 1: functional summary"):
        pdf = df[df["problem_id"] == prob_id].copy()

        total      = len(pdf)
        # .astype(bool) required: object-dtype booleans make ~ return -2/-1 in Python
        compiled   = pdf["compiled"].fillna(False).astype(bool)
        correct    = pdf["correctness"].fillna(False).astype(bool)

        n_compiled    = int(compiled.sum())
        n_correct     = int(correct.sum())
        n_wrong_build = int((compiled & ~correct).sum())
        n_failed      = int((~compiled).sum())

        # runtime stats — correct samples only
        cpdf = pdf[correct]
        rt_avg = float(cpdf["rt_mean"].mean())   if len(cpdf) > 0 else None
        rt_min = float(cpdf["rt_mean"].min())    if len(cpdf) > 0 else None
        rt_max = float(cpdf["rt_mean"].max())    if len(cpdf) > 0 else None
        rt_spread = (rt_max - rt_min)            if rt_min is not None else None

        # error diversity — bucket non-correct samples by error type
        fail_pdf = pdf[~correct]
        err_counts: dict = {}
        for _, row in fail_pdf.iterrows():
            key = str(row["error_name"] or "unknown")
            key = key.split("\n")[0][:80].strip()
            err_counts[key] = err_counts.get(key, 0) + 1

        if not err_counts:
            err_div = "no failures"
        elif len(err_counts) == 1:
            err_div = f"homogeneous [{next(iter(err_counts))}]"
        else:
            err_div = f"diverse ({len(err_counts)} error types)"

        rows1.append({
            "problem_id":      prob_id,
            "total":           total,
            "compiled":        n_compiled,
            "correct":         n_correct,
            "compiled_wrong":  n_wrong_build,
            "failed":          n_failed,
            "rt_avg_ms":       rt_avg,
            "rt_min_ms":       rt_min,
            "rt_max_ms":       rt_max,
            "rt_spread_ms":    rt_spread,
            "error_diversity": err_div,
            "error_counts":    json.dumps(err_counts),
        })

    s1 = pd.DataFrame(rows1)
    s1.to_csv(OUT_DIR / "stage1_per_problem.csv", index=False)

    # cross-check
    pak_path = RUN_DIR / "pass_at_k_results.json"
    if pak_path.exists():
        pak = json.loads(pak_path.read_text())
        pak_correct = (pak.get("metadata") or {}).get("total_correct_samples")
        our_correct = int(s1["correct"].sum())
        if pak_correct is not None:
            flag = "✅ MATCH" if pak_correct == our_correct else f"⚠️ MISMATCH (pass_at_k says {pak_correct})"
            print(f"Cross-check vs pass_at_k_results.json: {our_correct} correct — {flag}")

    # summary printout
    total_s = int(s1["total"].sum())
    print(f"\n=== Stage 1: Functional Summary ({len(s1)} problems) ===")
    print(f"  Total samples  : {total_s}")
    print(f"  Compiled       : {int(s1['compiled'].sum())} ({int(s1['compiled'].sum())/total_s:.1%})")
    print(f"  Correct        : {int(s1['correct'].sum())} ({int(s1['correct'].sum())/total_s:.1%})")
    print(f"  Compiled-wrong : {int(s1['compiled_wrong'].sum())}")
    print(f"  Failed/timeout : {int(s1['failed'].sum())}")

    div_col = s1["error_diversity"].str.extract(r"^(homogeneous|diverse|no failures)")[0]
    print("\n  Error diversity across problems:")
    for label, cnt in div_col.value_counts().items():
        print(f"    {label:<25}: {cnt} problems")

    # table
    disp = s1[["problem_id","total","compiled","correct","rt_avg_ms","rt_min_ms","rt_max_ms","error_diversity"]].copy()
    for c in ["rt_avg_ms","rt_min_ms","rt_max_ms"]:
        disp[c] = disp[c].map(lambda x: f"{x:.2f}" if x is not None and not (isinstance(x,float) and np.isnan(x)) else "—")
    disp = disp.sort_values(["correct","problem_id"], ascending=[False,True])
    print(f"\nPer-problem (first 20, sorted by #correct ↓):")
    print(disp.head(20).to_string(index=False))

    print(f"\nSaved: {OUT_DIR / 'stage1_per_problem.csv'}")
    _s1_done = True


Stage 1: functional summary: 100%|██████████| 100/100 [00:00<00:00, 804.31it/s]

Cross-check vs pass_at_k_results.json: 161 correct — ⚠️ MISMATCH (pass_at_k says 67)

=== Stage 1: Functional Summary (100 problems) ===
  Total samples  : 1000
  Compiled       : 884 (88.4%)
  Correct        : 161 (16.1%)
  Compiled-wrong : 723
  Failed/timeout : 116

  Error diversity across problems:
    diverse                  : 83 problems
    homogeneous              : 14 problems
    no failures              : 3 problems

Per-problem (first 20, sorted by #correct ↓):
 problem_id  total  compiled  correct rt_avg_ms rt_min_ms rt_max_ms                                       error_diversity
         19     10        10       10     17.82     17.70     18.10                                           no failures
         20     10        10       10     17.91     17.70     18.20                                           no failures
         30     10        10       10     17.93     17.70     18.20                                           no failures
         25     10        10    

## Stage 2 — AST Structural Similarity (Bigram Jaccard Distance)

Per-problem pairwise structural distance using **AST node bigrams** (parent→child type pairs).

**Why not APTED?** APTED is O(n³) in tree size. Triton kernels have 300–600 AST nodes → each pair takes
~10–60 s in pure Python; 45 pairs × 100 problems ≈ hours.

**Bigram Jaccard** is O(n) per kernel: walk the AST once, collect all (parent\_type, child\_type) pairs as a
multiset, then compare with Jaccard distance. Distance ∈ [0, 1]: 0 = identical structural vocabulary, 1 =
nothing in common. Runs in < 2 s for the full run folder.


In [4]:
# ─── Stage 2: AST Structural Similarity (Bigram Jaccard Distance) ────────────
#
# Was: APTED Tree-Edit Distance — O(n³) per pair, hours for 100 problems.
# Now: AST bigram Jaccard distance — O(n) per kernel, < 2 s total.
#
# A bigram is a (parent_node_type, child_node_type) pair.
# Jaccard distance = 1 − |A ∩ B| / |A ∪ B| on the bigram multisets.
# ─────────────────────────────────────────────────────────────────────────────

if not RUN_STAGE[2]:
    print("Stage 2 skipped (RUN_STAGE[2]=False)")
else:
    from collections import Counter

    def ast_bigrams(code: str):
        """Return Counter of (parent_type, child_type) pairs, or None on SyntaxError."""
        try:
            tree = ast.parse(code)
        except SyntaxError:
            return None
        bg = Counter()
        for node in ast.walk(tree):
            pt = type(node).__name__
            for child in ast.iter_child_nodes(node):
                bg[(pt, type(child).__name__)] += 1
        return bg

    def bigram_jaccard(a, b):
        """Jaccard distance ∈ [0, 1]; None if either bag is missing."""
        if a is None or b is None:
            return None
        inter = sum((a & b).values())
        union = sum((a | b).values())
        return 1.0 - inter / union if union > 0 else 0.0

    # ── step 1: parse all kernels once (one AST walk per kernel) ─────────────
    all_bigrams: dict = {}
    n_parse_fail = 0
    for prob_id in tqdm(sorted(problems.keys()), desc="Stage 2: parsing ASTs"):
        all_bigrams[prob_id] = {}
        for sid, data in problems[prob_id].items():
            bg = ast_bigrams(data["code"])
            all_bigrams[prob_id][sid] = bg
            if bg is None:
                n_parse_fail += 1

    # ── step 2: pairwise Jaccard distance matrix per problem ─────────────────
    stage2_results = {}
    rows2 = []

    for prob_id in tqdm(sorted(problems.keys()), desc="Stage 2: pairwise distances"):
        sids = sorted(problems[prob_id].keys())
        bags = [all_bigrams[prob_id][s] for s in sids]
        n    = len(sids)

        mat = np.full((n, n), np.nan)
        np.fill_diagonal(mat, 0.0)
        for i in range(n):
            for j in range(i + 1, n):
                d = bigram_jaccard(bags[i], bags[j])
                mat[i, j] = mat[j, i] = d if d is not None else np.nan

        pf = sum(1 for b in bags if b is None)
        stage2_results[prob_id] = {"matrix": mat, "sample_ids": sids, "parse_fail": pf}

        upper = mat[np.triu_indices(n, k=1)]
        valid = upper[~np.isnan(upper)]
        mean_dist = float(np.mean(valid)) if len(valid) else np.nan
        rows2.append({
            "problem_id":       prob_id,
            "n_samples":        n,
            "parse_fail":       pf,
            "mean_pairwise_dist": mean_dist,
        })

    # ── save ─────────────────────────────────────────────────────────────────
    np.savez_compressed(OUT_DIR / "stage2_ast_struct.npz",
                        **{str(k): v["matrix"] for k, v in stage2_results.items()})

    s2_df = pd.DataFrame(rows2).sort_values("problem_id")
    s2_df.to_csv(OUT_DIR / "stage2_ast_struct_summary.csv", index=False)

    valid = s2_df["mean_pairwise_dist"].dropna()
    print(f"\n=== Stage 2: AST Structural Distance (Bigram Jaccard) ===")
    print(f"  Parse failures            : {n_parse_fail} samples")
    print(f"  Distance range            : 0 = identical vocabulary, 1 = nothing in common")
    print(f"  Global mean pairwise dist : {valid.mean():.4f}")
    most_alike   = s2_df.loc[s2_df["mean_pairwise_dist"].idxmin(), "problem_id"]
    most_diverse = s2_df.loc[s2_df["mean_pairwise_dist"].idxmax(), "problem_id"]
    print(f"  Most structurally alike   (min dist) : problem {most_alike:.0f}")
    print(f"  Most structurally diverse (max dist) : problem {most_diverse:.0f}")
    print(f"\nSaved: {OUT_DIR / 'stage2_ast_struct.npz'}")
    print(f"Saved: {OUT_DIR / 'stage2_ast_struct_summary.csv'}")
    _s2_done = True


Stage 2: pairwise distances: 100%|██████████| 100/100 [00:00<00:00, 679.71it/s]


=== Stage 2: AST Structural Distance (Bigram Jaccard) ===
  Parse failures            : 15 samples
  Distance range            : 0 = identical vocabulary, 1 = nothing in common
  Global mean pairwise dist : 0.2867
  Most structurally alike   (min dist) : problem 19
  Most structurally diverse (max dist) : problem 92

Saved: /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis/stage2_ast_struct.npz
Saved: /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis/stage2_ast_struct_summary.csv


## Stage 3 — Code-Embedding Cosine Similarity
Encodes each kernel with `jinaai/jina-code-embeddings-1.5b`; computes pairwise cosine similarity within each problem. Embeddings are cached to disk.
Requires: `pip install torch sentence-transformers`

In [5]:
# ─── Stage 3: Code-Embedding Cosine Similarity ───────────────────────────────
if not RUN_STAGE[3]:
    print("Stage 3 skipped (RUN_STAGE[3]=False)")
else:
    try:
        import torch
        from sentence_transformers import SentenceTransformer
    except ImportError:
        print("❌ Missing torch / sentence-transformers.")
        print("   Run: pip install torch sentence-transformers")
        raise

    CACHE_PATH = OUT_DIR / "embeddings.npz"

    # build ordered key list
    all_keys  = [(pid, sid)
                 for pid in sorted(problems)
                 for sid in sorted(problems[pid])]
    all_codes = [problems[pid][sid]["code"] for pid, sid in all_keys]

    # ── load or compute embeddings ────────────────────────────────────────────
    if CACHE_PATH.exists():
        _cache       = np.load(CACHE_PATH, allow_pickle=True)
        cached_keys  = [tuple(k) for k in _cache["keys"]]
        cached_embs  = _cache["embeddings"]
        print(f"Loaded {len(cached_keys)} cached embeddings from {CACHE_PATH.name}")
    else:
        cached_keys, cached_embs = [], None

    cached_set   = set(map(tuple, cached_keys))
    new_keys     = [k for k in all_keys  if k not in cached_set]
    new_codes    = [c for k, c in zip(all_keys, all_codes) if k not in cached_set]

    if new_keys:
        print(f"Computing {len(new_keys)} embeddings on {DEVICE} (model: {EMBED_MODEL}) …")
        _model   = SentenceTransformer(EMBED_MODEL, trust_remote_code=True, device=DEVICE)
        new_embs = _model.encode(
            new_codes,
            show_progress_bar=True,          # tqdm bar from sentence-transformers
            batch_size=4 if DEVICE == "cpu" else 32,
            convert_to_numpy=True,
            normalize_embeddings=True,       # unit norm → cosine = dot product
        )
        del _model   # free memory

        # merge with cache
        all_emb_keys = cached_keys + new_keys
        all_embs     = (np.vstack([cached_embs, new_embs]) if cached_embs is not None
                        else new_embs)

        # save updated cache
        np.savez_compressed(CACHE_PATH,
                            keys=np.array(all_emb_keys, dtype=object),
                            embeddings=all_embs)
        print(f"Saved embeddings cache → {CACHE_PATH.name}")
    else:
        all_emb_keys = cached_keys
        all_embs     = cached_embs
        print("All embeddings already cached — skipping model inference.")

    # reorder to match all_keys
    k2e = {tuple(k): e for k, e in zip(all_emb_keys, all_embs)}
    embeddings = np.stack([k2e[k] for k in all_keys])   # (N, dim), unit-normed
    k2i = {k: i for i, k in enumerate(all_keys)}

    # ── per-problem cosine similarity matrices ────────────────────────────────
    stage3_results = {}
    rows3          = []
    all_cos        = []

    for prob_id in tqdm(sorted(problems), desc="Stage 3: cosine matrices"):
        sids = sorted(problems[prob_id])
        idxs = [k2i[(prob_id, s)] for s in sids]
        embs = embeddings[idxs]                    # (n, dim)
        sim  = np.clip(embs @ embs.T, -1.0, 1.0)  # cosine sim matrix

        stage3_results[prob_id] = {"matrix": sim, "sample_ids": sids}

        n = len(sids)
        if n > 1:
            upper = sim[np.triu_indices(n, k=1)]
            mean_cos = float(np.mean(upper))
            min_cos  = float(np.min(upper))
            max_cos  = float(np.max(upper))
            all_cos.extend(upper.tolist())
        else:
            mean_cos = min_cos = max_cos = np.nan

        rows3.append({"problem_id": prob_id, "n_samples": n,
                      "mean_cosine_sim": mean_cos,
                      "min_cosine_sim":  min_cos,
                      "max_cosine_sim":  max_cos})

    s3_df = pd.DataFrame(rows3).sort_values("problem_id")
    s3_df.to_csv(OUT_DIR / "stage3_embedding_sim_summary.csv", index=False)
    np.savez_compressed(OUT_DIR / "stage3_cosine_matrices.npz",
                        **{str(k): v["matrix"] for k, v in stage3_results.items()})

    valid_cos = s3_df["mean_cosine_sim"].dropna()
    print(f"\n=== Stage 3: Code-Embedding Cosine Similarity ===")
    print(f"  Global mean pairwise cosine sim : {valid_cos.mean():.4f}")
    most_alike   = s3_df.loc[s3_df["mean_cosine_sim"].idxmax(), "problem_id"]
    most_diverse = s3_df.loc[s3_df["mean_cosine_sim"].idxmin(), "problem_id"]
    print(f"  Most semantically alike   (max cosine) : problem {most_alike:.0f}")
    print(f"  Most semantically diverse (min cosine) : problem {most_diverse:.0f}")
    print(f"\nSaved: {OUT_DIR / 'stage3_embedding_sim_summary.csv'}")
    print(f"Saved: {OUT_DIR / 'stage3_cosine_matrices.npz'}")
    _s3_done = True


Stage 3 skipped (RUN_STAGE[3]=False)


## Aggregate Report
Combines all stage outputs into one summary CSV + scatter plot.

In [6]:
# ─── Aggregate Report ─────────────────────────────────────────────────────────
print("=== Aggregate Report ===\n")

agg_rows = []
for prob_id in sorted(problems.keys()):
    row = {"problem_id": prob_id}

    if s1 is not None:
        r1 = s1[s1["problem_id"] == prob_id]
        if len(r1):
            row.update({k: r1[k].values[0] for k in
                        ["total","compiled","correct","compiled_wrong","failed",
                         "rt_avg_ms","rt_min_ms","rt_max_ms","error_diversity"]})

    if s2_df is not None:
        r2 = s2_df[s2_df["problem_id"] == prob_id]
        if len(r2): row["mean_ast_dist"] = r2["mean_pairwise_dist"].values[0]

    if s3_df is not None:
        r3 = s3_df[s3_df["problem_id"] == prob_id]
        if len(r3): row["mean_embed_cos"] = r3["mean_cosine_sim"].values[0]

    agg_rows.append(row)

summary_df = pd.DataFrame(agg_rows)
summary_df.to_csv(OUT_DIR / "summary.csv", index=False)
print(f"Full summary saved → {OUT_DIR / 'summary.csv'}")

# ── summary table ─────────────────────────────────────────────────────────────
disp_cols = [c for c in ["problem_id","total","correct","rt_avg_ms",
                          "mean_ast_dist","mean_embed_cos","error_diversity"]
             if c in summary_df.columns]
s_disp = summary_df[disp_cols].copy()
for c in ["rt_avg_ms"]:
    if c in s_disp:
        s_disp[c] = s_disp[c].map(lambda x: f"{x:.2f}" if pd.notna(x) else "—")
for c in ["mean_ast_dist","mean_embed_cos"]:
    if c in s_disp:
        s_disp[c] = s_disp[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "—")

sort_col = "correct" if "correct" in s_disp.columns else "problem_id"
print(f"\nTop 20 (sorted by #{sort_col} ↓):")
print(s_disp.sort_values(sort_col, ascending=False).head(20).to_string(index=False))

# ── per-problem detailed plots ────────────────────────────────────────────────
per_prob_dir = OUT_DIR / "per_problem"
per_prob_dir.mkdir(exist_ok=True)

# pre-index per-problem df slices once for speed
_prob_slices = {pid: df[df["problem_id"] == pid].copy() for pid in sorted(problems.keys())}

print(f"\nGenerating per-problem detail plots → {per_prob_dir}/")

for prob_id in tqdm(sorted(problems.keys()), desc="Per-problem plots"):
    pdf      = _prob_slices[prob_id]
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle(
        f"Problem {prob_id}  (level {pdf['level'].iloc[0]}  ·  {len(pdf)} samples)",
        fontsize=13, fontweight="bold",
    )
    ax_status, ax_runtime, ax_ted, ax_cos = axes[0,0], axes[0,1], axes[1,0], axes[1,1]

    # ── Panel 1: Status bar ───────────────────────────────────────────────────
    if s1 is not None:
        r1 = s1[s1["problem_id"] == prob_id]
        if len(r1):
            r      = r1.iloc[0]
            labels = ["Correct", "Compiled\n(wrong)", "Failed /\ntimeout"]
            vals   = [r["correct"], r["compiled_wrong"], r["failed"]]
            colors = ["#4CAF50", "#FF9800", "#F44336"]
            bars   = ax_status.bar(labels, vals, color=colors, edgecolor="white",
                                   linewidth=0.8, width=0.5)
            for bar, v in zip(bars, vals):
                if v > 0:
                    ax_status.text(bar.get_x() + bar.get_width() / 2,
                                   bar.get_height() + 0.05,
                                   str(int(v)), ha="center", va="bottom", fontsize=11,
                                   fontweight="bold")
            ax_status.set_ylim(0, int(r["total"]) + 1.5)
            ax_status.set_ylabel("# samples", fontsize=9)
            ax_status.set_title(f"Functional status  (n={int(r['total'])})", fontsize=10)
            ax_status.tick_params(axis="x", labelsize=9)
            ax_status.spines[["top","right"]].set_visible(False)
        else:
            ax_status.text(0.5, 0.5, "No Stage 1 data", ha="center", va="center",
                           transform=ax_status.transAxes)
            ax_status.set_title("Functional status", fontsize=10)
    else:
        ax_status.text(0.5, 0.5, "Stage 1 not run", ha="center", va="center",
                       transform=ax_status.transAxes, color="grey")
        ax_status.set_title("Functional status", fontsize=10)

    # ── Panel 2: Runtime per correct sample (bar + error bar for std) ─────────
    if s1 is not None:
        correct_mask = pdf["correctness"].fillna(False).astype(bool)
        cpdf = pdf[correct_mask].sort_values("sample_id")
        if len(cpdf) > 0 and cpdf["rt_mean"].notna().any():
            x    = [f"S{int(sid)}" for sid in cpdf["sample_id"]]
            y    = cpdf["rt_mean"].fillna(0).values
            yerr = cpdf["rt_std"].fillna(0).values
            ax_runtime.bar(x, y, yerr=yerr, capsize=5,
                           color="steelblue", alpha=0.82, edgecolor="navy", linewidth=0.5,
                           error_kw={"elinewidth": 1.2, "ecolor": "navy", "capthick": 1.2})
            ax_runtime.set_xlabel("Sample", fontsize=9)
            ax_runtime.set_ylabel("Runtime mean (ms)", fontsize=9)
            ax_runtime.set_title("Runtime per correct sample (ms ± std)", fontsize=10)
            ax_runtime.tick_params(axis="x", labelsize=8, rotation=30)
            ax_runtime.spines[["top","right"]].set_visible(False)
        else:
            ax_runtime.text(0.5, 0.5, "No correct samples", ha="center", va="center",
                            transform=ax_runtime.transAxes, color="grey")
            ax_runtime.set_title("Runtime per correct sample", fontsize=10)
    else:
        ax_runtime.text(0.5, 0.5, "Stage 1 not run", ha="center", va="center",
                        transform=ax_runtime.transAxes, color="grey")
        ax_runtime.set_title("Runtime per correct sample", fontsize=10)

    # ── Panel 3: AST structural distance heatmap ──────────────────────────────
    if stage2_results is not None and prob_id in stage2_results:
        res2      = stage2_results[prob_id]
        labels2   = [f"S{s}" for s in res2["sample_ids"]]
        row2      = s2_df[s2_df["problem_id"] == prob_id]
        mean_dist = float(row2["mean_pairwise_dist"].values[0]) if len(row2) else float("nan")
        annot     = len(labels2) <= 12
        sns.heatmap(res2["matrix"], ax=ax_ted, vmin=0, vmax=1, cmap="YlOrRd",
                    xticklabels=labels2, yticklabels=labels2,
                    annot=annot, fmt=".2f", annot_kws={"size": 7},
                    cbar_kws={"shrink": 0.8, "label": "struct. dist."})
        ax_ted.set_title(f"AST Structural Distance  (mean={mean_dist:.4f})", fontsize=10)
        ax_ted.tick_params(labelsize=8)
    else:
        ax_ted.text(0.5, 0.5, "Stage 2 not run", ha="center", va="center",
                    transform=ax_ted.transAxes, color="grey")
        ax_ted.set_title("AST Structural Distance", fontsize=10)

    # ── Panel 4: Cosine similarity heatmap ────────────────────────────────────
    if stage3_results is not None and prob_id in stage3_results:
        res3    = stage3_results[prob_id]
        labels3 = [f"S{s}" for s in res3["sample_ids"]]
        row3    = s3_df[s3_df["problem_id"] == prob_id]
        cos_val = float(row3["mean_cosine_sim"].values[0]) if len(row3) else float("nan")
        annot   = len(labels3) <= 12
        sns.heatmap(res3["matrix"], ax=ax_cos, vmin=0, vmax=1, cmap="RdYlGn",
                    xticklabels=labels3, yticklabels=labels3,
                    annot=annot, fmt=".2f", annot_kws={"size": 7},
                    cbar_kws={"shrink": 0.8, "label": "cosine sim"})
        ax_cos.set_title(f"Embedding Cosine Similarity  (mean={cos_val:.4f})", fontsize=10)
        ax_cos.tick_params(labelsize=8)
    else:
        ax_cos.text(0.5, 0.5, "Stage 3 not run", ha="center", va="center",
                    transform=ax_cos.transAxes, color="grey")
        ax_cos.set_title("Embedding Cosine Similarity", fontsize=10)

    plt.tight_layout()
    out_path = per_prob_dir / f"problem_{prob_id:03d}.png"
    plt.savefig(out_path, bbox_inches="tight", dpi=100)
    plt.close(fig)   # close to avoid flooding notebook with 100 inline figures

print(f"\nSaved {len(problems)} per-problem plots → {per_prob_dir}/")
print("\n✅ Done. All artifacts written to:", OUT_DIR)


=== Aggregate Report ===

Full summary saved → /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis/summary.csv

Top 20 (sorted by #correct ↓):
 problem_id  total  correct rt_avg_ms mean_ast_dist                                       error_diversity
         19     10       10     17.82        0.0000                                           no failures
         30     10       10     17.93        0.0152                                           no failures
         20     10       10     17.91        0.0119                                           no failures
         25     10        9     17.96        0.0762                                     homogeneous [nan]
         28     10        8     17.82        0.0511                                     homogeneous [nan]
         31     10        8     17.94        0.0748 homogeneous [triton.compiler.errors.CompilationError]
          5     10        8     11.05        0.2304           

Per-problem plots: 100%|██████████| 100/100 [00:48<00:00,  2.06it/s]


Saved 100 per-problem plots → /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis/per_problem/

✅ Done. All artifacts written to: /home/jan-bosenius/PycharmProjects/GuidedResearch/notebooks/runs/Qwen3-Coder-Next_level1_triton/analysis
